# Extract Items to CSV (Using buff_ui and Improved Source Discovery)

This notebook extracts all items from Anno 117 into a CSV file with the following columns:
- guid
- name
- rarity
- trade_price
- targets (comma-separated names)
- buffs (descriptions of all buff attributes)
- boost_condition (for ItemWithBoost: condition when boost activates)
- boost_buffs (for ItemWithBoost: boosted buff attributes)
- source (where the item can be obtained)

**Features:**
- Uses the new `buff_ui` property for simplified buff extraction
- Improved source discovery covering 7 source types:
  - Traders (Selling)
  - Research (Tech Tree) - no hardcoded GUIDs
  - Quests - traversing decision chains
  - NPC Drops (Rivals, Emperor, Pirates)
  - Achievements
  - Festivals
  - Functions & Triggers (progression unlocks)
- Returns Text objects for multi-language support
- Uses efficient reverse reference system (`in_reward_pool`, `referenced_by`)

In [1]:
from assetextractor.extraction.utils import Config
from assetextractor.parsing.core.assets import Asset, AssetCache
from assetextractor.parsing.core.templates import Template
from assetextractor.parsing.core.texts import StandardTextConverter, Text


import csv
from pathlib import Path
import typing as t
import pandas as pd

In [2]:
MAX_CONTRACT_SCORE = 3600 # see template Participant 3rdParty.ContractProvider.DestroyContractBalancing[16].MaxAmount
LANGUAGE = "english"

In [3]:
# Load asset cache

config = Config.from_json("config.json")
assets = AssetCache.load(config)
templates = assets.templates
texts = assets.texts
texts.converter = StandardTextConverter(LANGUAGE)

print(f"Loaded assets successfully")

Loaded assets successfully


In [4]:
# Helper Functions

def flatten_pool(pool: Asset | None) -> list[int]:
    """Recursively flatten an AssetPool into individual asset GUIDs."""
    if pool is None:
        return []

    if "AssetPool" not in pool.template.name:
        return [pool.guid]

    res = []
    for entry in pool.AssetPool.AssetList:
        if entry.Asset():  # Last item can sometimes be None
            res += flatten_pool(entry.Asset())
    return res


def get_localized_name(asset: Asset) -> str:
    """Get the name of an asset from localization."""
    if asset.text is not None:
        return asset.text()
    # Fallback to internal name
    return asset.name if asset.name else f"Asset_{asset.guid}"



def get_target_names(target_guids: list[int]) -> str:
    """Convert list of target GUIDs to comma-separated names."""
    names = []
    for guid in target_guids:
        try:
            target_asset = assets[guid]
            if target_asset:
                names.append(get_localized_name(target_asset))
        except:
            names.append(f"Unknown_{guid}")
    return ", ".join(names) if names else ""


def format_buff_ui(buff_ui) -> str:
    """Format a single BuffUI object to text."""
    parts = []
    
    # Get text
    if buff_ui.text:
        if isinstance(buff_ui.text, Text):
            text = buff_ui.text()
        else:
            text = str(buff_ui.text)
        
        if text:
            parts.append(text)
    
    # Get value
    if buff_ui.value:
        parts.append(buff_ui.value)
    
    return ": ".join(parts) if parts else ""


def format_buff_attributes(buff_asset: Asset, visited_effects: set[int] = None) -> str:
    """Extract and format buff attributes using buff_ui property."""
    if visited_effects is None:
        visited_effects = set()
    
    functional_effects = []
    
    # Get buff UI from the asset
    buff_ui_list = buff_asset.buff_ui
    
    # Format all BuffUI objects
    buff_texts = []
    for buff_ui in buff_ui_list:
        formatted = format_buff_ui(buff_ui)
        if formatted:
            buff_texts.append(formatted)
    
    # Combine functional effects and direct buffs
    all_texts = functional_effects + buff_texts
    
    return "; ".join(all_texts) if all_texts else "No attributes"


def extract_boost_buffs(item_asset: Asset, visited_effects: set[int] = None) -> str:
    """Extract and format boost buff attributes from an ItemWithBoost asset."""
    if visited_effects is None:
        visited_effects = set()
    
    try:
        boost_buffs_attr = item_asset.find("ItemWithBoost.BoostBuffs")
        if not boost_buffs_attr:
            return ""
        
        buff_descriptions = []
        for buff_entry in boost_buffs_attr:
            try:
                buff_guid = buff_entry.GUID.guid
                if buff_guid:
                    buff_asset = assets[buff_guid]
                    if buff_asset:
                        buff_desc = format_buff_attributes(buff_asset, visited_effects)
                        if buff_desc and buff_desc != "No attributes":
                            buff_descriptions.append(buff_desc)
            except:
                pass
        
        return " | ".join(buff_descriptions) if buff_descriptions else ""
    except:
        return ""


print("Helper functions loaded")

Helper functions loaded


In [5]:
# Item Source Tracking (Improved - Returns Text objects)

# Text IDs for source type labels
from assetextractor.parsing.core.attributes import ListAttribute
from assetextractor.parsing.core.texts import Text


SOURCE_TEXT_IDS = {
    "quest": -6905698394117185352,      # "Quest"
    "selling": -6902222124635972240,    # "Selling"
    "contracts": -6914021190765224130,  # "Contracts"
    "research": -6902138578600598283,   # "Research"
    "drops": -6917297453044695070,      # "Flotsam" (repurposed for Drops)
    "festival": -6908773579491322283
}


def get_source_display_name(source: Asset) -> Text | None:
    """Get localized display name for a source asset (returns Text object or string)."""
    # Prefer localized text
    if source.text:
        return source.text
    
    # Check for TechName (for Tech template)
    if source.template.name == "Tech":
        tech_name_attr = source.find("Tech.TechName")
        if tech_name_attr:
            text_ref = tech_name_attr()
            if text_ref and isinstance(text_ref, int):
                text_obj = texts.elements.get(text_ref)
                if text_obj:
                    return text_obj
    
    # Fallback to internal name (as string)
    return source.name


def _traverse_to_quest(start_asset: Asset, visited: set, max_depth: int = 10) -> Asset | None:
    """Traverse upward through references to find a quest."""
    current = start_asset
    depth = 0
    
    while depth < max_depth:
        # Check if current asset is a quest
        template = current.template.name
        if template == "Quest" or ("Quest" in template and "Component" not in template):
            return current

        linked_quest = current.find_ref("Objective.Objective.ConditionQuestObjective.LinkedQuestEntry")
        if linked_quest is not None:
            return linked_quest
        
        # Find next asset in chain
        found_next = False
        for ref_guid, weighted_ref in current.referenced_by.items():
            if ref_guid not in visited:
                visited.add(ref_guid)
                current = weighted_ref.source
                found_next = True
                break
        
        if not found_next:
            break
        
        depth += 1
    
    return None

def _process_contract_rewards(trader: Asset, item: Asset, processed_pools: set[Asset]):
    reward_list = trader.find("ContractProvider.ItemRewards")
    if not isinstance(reward_list, ListAttribute):
        return None

    probability = 0
    for element in reward_list:
        lower = min(element.MinRange(), MAX_CONTRACT_SCORE)
        upper = min(element.MaxRange(), MAX_CONTRACT_SCORE)

        pool = element.find_ref("ItemRewardPool")
        
        if pool is None:
            continue

        probability += (upper - lower)/MAX_CONTRACT_SCORE * pool.pool_assets().get(item, 0)

        processed_pools.add(pool)

    return probability


def find_all_item_sources(item: Asset, assets: AssetCache) -> dict[str, list[dict]]:
    """Discover all sources for an item across all mechanisms."""
    sources = {
        "selling": [],
        "contracts": [],
        "research": [],
        "quest": [],
        "drops": [],
        "achievement": [],
        "festival": [],
        "function": [],
        "trigger": []
    }
    
    processed_pools = set[Asset]()

    # Check pool-based sources
    if hasattr(item, 'in_reward_pool'):
        for pool_guid, ref in item.in_reward_pool.items():
            pool = assets[pool_guid]

            if pool in processed_pools:
                continue
            
            # Check all assets referencing this pool
            for pool_ref in pool.referenced_by.values():
                source = pool_ref.source
                template = source.template.name
                
                # Categorize by template
                if "Participant" in template and "3rdParty" in template:
                    # Check if it's drops, contract or selling
                    if "ShipDropRewardPool" in pool_ref.path:
                        sources["drops"].append({
                            "name": get_source_display_name(source),
                            "guid": source.guid,
                            "probability": ref.weight
                        })
                    elif "OfferedItems" in pool_ref.path:
                        sources["selling"].append({
                            "name": get_source_display_name(source),
                            "guid": source.guid,
                            "probability": ref.weight
                        })
                    elif "ContractProvider" in pool_ref.path:
                        prob = _process_contract_rewards(source, item, processed_pools)
                        if prob is not None:
                            sources["contracts"].append({
                                "name": get_source_display_name(source),
                                "guid": source.guid,
                                "probability": prob
                            })
                
                elif template == "Tech":
                    sources["research"].append({
                        "name": get_source_display_name(source),
                        "guid": source.guid,
                        "probability": ref.weight
                    })
                
                elif template == "Achievement":
                    sources["achievement"].append({
                        "name": get_source_display_name(source),
                        "guid": source.guid,
                        "probability": ref.weight
                    })
                
                elif template == "Festival":
                    sources["festival"].append({
                        "name": get_source_display_name(source),
                        "guid": source.guid,
                        "probability": ref.weight
                    })
                
                elif template == "Function":
                    sources["function"].append({
                        "name": get_source_display_name(source),
                        "guid": source.guid,
                        "probability": ref.weight
                    })
                
                elif template in ("Trigger", "Gate"):
                    sources["trigger"].append({
                        "name": get_source_display_name(source),
                        "guid": source.guid,
                        "probability": ref.weight
                    })
                
                elif "Objective" in template:
                    # Handle quest objectives (traverse to find parent quest)
                    visited = set()
                    quest = _traverse_to_quest(source, visited)
                    if quest:
                        sources["quest"].append({
                            "name": get_source_display_name(quest),
                            "guid": quest.guid,
                            "objective": source.guid,
                            "probability": ref.weight
                        })
    
    # Check reference-based sources (quest sequences)
    if hasattr(item, 'referenced_by'):
        visited = set()
        for ref_guid, weighted_ref in item.referenced_by.items():
            source = weighted_ref.source
            
            if "Sequence" in source.template.name and "ActionAddGoodsToItemContainer" in weighted_ref.path:
                quest = _traverse_to_quest(source, visited)
                if quest:
                    # Check if not already added
                    already_added = any(s["guid"] == quest.guid for s in sources["quest"])
                    if not already_added:
                        sources["quest"].append({
                            "name": get_source_display_name(quest),
                            "guid": quest.guid,
                            "sequence": source.guid
                        })
    
    return sources


def format_source_for_csv(source_type: str, source_name, probability: float | None = None) -> str:
    """Format a single source for CSV export (uses localized text)."""
    # Get source type label
    type_labels = {
        "selling": "Selling",
        "contracts": "Contracts",
        "research": "Research",
        "quest": "Quest",
        "drops": "Drops",
        "achievement": "Achievement",
        "festival": "Festival",
        "function": "Event",
        "trigger": "Unlock"
    }
    
    text_id = SOURCE_TEXT_IDS.get(source_type, None)
    if text_id is not None:
        label = assets.texts.get(text_id)()
    else:
        label = type_labels.get(source_type, source_type.title())
    
    # Convert Text object to localized string
    if hasattr(source_name, "values"):
        name_str = source_name()
    else:
        name_str = str(source_name) if source_name else "Unknown"
    
    # Add probability if significant
    if probability and probability < 1.0:
        return f"{label}: {name_str} ({probability:.3%})"
    else:
        return f"{label}: {name_str}"


def format_all_sources_csv(sources: dict[str, list[dict]]) -> str:
    """Format all sources as semicolon-separated string for CSV."""
    all_sources = []
    
    # Order by importance
    # exclude achievement, function, and trigger as these are no sources
    order = ["selling", "contracts", "research", "festival", "quest", "drops"]
    
    # Deduplicate by GUID
    for source_type in order:
        seen_guids = set()
        for source in sources[source_type]:
            guid = source["guid"]
            if guid not in seen_guids:
                seen_guids.add(guid)
                formatted = format_source_for_csv(source_type, source["name"], source.get("probability"))
                all_sources.append(formatted)
    
    return "; ".join(all_sources) if all_sources else ""


print("Improved item source tracking loaded")

Improved item source tracking loaded


In [6]:
# Boost Condition Extraction (Complete - All condition types)

def extract_boost_condition(item_asset: Asset) -> str:
    """Extract the boost condition from an ItemWithBoost asset."""
    try:
        condition_attr = item_asset.find("ItemWithBoost.BoostCondition.PreConditionList.Condition")
        if not condition_attr:
            return ""

        condition = condition_attr
        if not condition:
            return ""

        # ConditionAlwaysTrue
        try:
            if hasattr(condition, 'ConditionAlwaysTrue'):
                has_other = any(hasattr(condition, ct) for ct in [
                    'ConditionObjectCount', 'ConditionDominantPatron', 'ConditionNeedAttributeCounter',
                    'ConditionPlayerCounter', 'ConditionActiveEmperor', 'ConditionReligion',
                    'ConditionMonumentEventsActive', 'ConditionEmperorRelation', 'ConditionDiplomacyState',
                    'ConditionItemUsed', 'ConditionWarState', 'ConditionInStorage', 'ConditionTradeRouteCount'
                ])
                if not has_other:
                    return "Always active"
        except:
            pass

        # ConditionObjectCount
        try:
            if hasattr(condition, 'ConditionObjectCount'):
                amount = condition.ConditionObjectCount.Amount()
                comparison_op = condition.ConditionObjectCount.ComparisonOp()
                comparison_map = {0: ">=", "AtLeast": ">=", "AtMost": "<=", "LessThan": "<", "GreaterThan": ">", "Equal": "="}
                op_symbol = comparison_map.get(comparison_op, ">=")
                obj_guid = condition.ObjectFilter.ObjectGUID()
                if obj_guid:
                    obj_name = get_localized_name(obj_guid)
                    amount_str = str(int(amount)) if amount == int(amount) else str(amount)
                    return f"{obj_name} {op_symbol} {amount_str}"
        except:
            pass

        # ConditionNeedAttributeCounter
        try:
            if hasattr(condition, 'ConditionNeedAttributeCounter'):
                need_type = condition.ConditionNeedAttributeCounter.NeedAttributeType()
                if need_type == 0 or need_type == "0":
                    need_type = "All Need Attributes"
                amount = condition.ConditionNeedAttributeCounter.NeedAttributeAmount()
                is_global = condition.ConditionNeedAttributeCounter.UseGlobalSum()

                if need_type and amount:
                    amount_str = str(int(amount)) if amount == int(amount) else str(amount)
                    result = f"{need_type} >= {amount_str}"
                    if is_global:
                        result += " (Global)"
                    return result
        except:
            pass

        # ConditionDominantPatron
        try:
            if hasattr(condition, 'ConditionDominantPatron'):
                patron_guid = condition.ConditionDominantPatron.PatronGUID()
                if patron_guid:
                    return f"Dominant Patron: {get_localized_name(patron_guid)}"
        except:
            pass

        # ConditionReligion
        try:
            if hasattr(condition, 'ConditionReligion'):
                religion_asset = condition.ConditionReligion.ReligionAsset()
                if religion_asset:
                    return f"Patron: {get_localized_name(religion_asset)}"
        except:
            pass

        # ConditionPlayerCounter
        try:
            if hasattr(condition, 'ConditionPlayerCounter'):
                player_counter = condition.ConditionPlayerCounter.PlayerCounter()
                comparison_op = condition.ConditionPlayerCounter.ComparisonOp()
                counter_amount = condition.ConditionPlayerCounter.CounterAmount()
                comparison_map = {0: ">=", "AtLeast": ">=", "AtMost": "<=", "LessThan": "<", "GreaterThan": ">", "Equal": "="}
                op_symbol = comparison_map.get(comparison_op, ">=")
                scope = condition.ConditionPlayerCounter.CounterScope()
                if scope == 0 or scope == "0":
                    scope = "Global"

                context_building = condition.ConditionPlayerCounter.Context()
                if context_building:
                    building_name = get_localized_name(context_building)
                    amount_str = str(int(counter_amount)) if counter_amount == int(counter_amount) else str(counter_amount)
                    return f"{building_name} {op_symbol} {amount_str} ({scope})"

                if player_counter and player_counter != 0:
                    counter_name = str(player_counter)
                    amount_str = str(int(counter_amount)) if counter_amount == int(counter_amount) else str(counter_amount)
                    return f"{counter_name} {op_symbol} {amount_str} ({scope})"
        except:
            pass

        # ConditionActiveEmperor
        try:
            if hasattr(condition, 'ConditionActiveEmperor'):
                emperor = condition.ConditionActiveEmperor.EmperorParticipant()
                if emperor:
                    return f"Emperor: {get_localized_name(emperor)}"
        except:
            pass

        # ConditionEmperorRelation
        try:
            if hasattr(condition, 'ConditionEmperorRelation'):
                return "Emperor relation required"
        except:
            pass

        # ConditionDiplomacyState
        try:
            if hasattr(condition, 'ConditionDiplomacyState'):
                profile2 = condition.ConditionDiplomacyState.Profile2()
                desired_state = condition.ConditionDiplomacyState.DesiredState()
                if profile2 and desired_state:
                    profile_name = get_localized_name(profile2)
                    return f"Diplomacy with {profile_name}: {desired_state}"
        except:
            pass

        # ConditionTradeRouteCount
        try:
            if hasattr(condition, 'ConditionTradeRouteCount'):
                count = condition.ConditionTradeRouteCount.TradeRouteCount()
                count_op = condition.ConditionTradeRouteCount.CountComparisonOp()
                comparison_map = {0: ">=", "AtLeast": ">=", "AtMost": "<=", "LessThan": "<", "GreaterThan": ">", "Equal": "="}
                op_symbol = comparison_map.get(count_op, ">=")
                if count:
                    return f"Trade routes {op_symbol} {int(count)}"
        except:
            pass

        # ConditionItemUsed
        try:
            if hasattr(condition, 'ConditionItemUsed'):
                item_amount = condition.ConditionItemUsed.ItemAmount()
                if item_amount:
                    return f"{int(item_amount)} items equipped"
        except:
            pass

        # ConditionMonumentEventsActive
        try:
            if hasattr(condition, 'ConditionMonumentEventsActive'):
                return "Monument events active"
        except:
            pass

        # ConditionWarState
        try:
            if hasattr(condition, 'ConditionWarState'):
                return "At war"
        except:
            pass

        # ConditionInStorage
        try:
            if hasattr(condition, 'ConditionInStorage'):
                return "Items in storage"
        except:
            pass

        return "Boost condition active"
    except:
        return ""


print("Boost condition extraction loaded")

Boost condition extraction loaded


In [7]:
# Extract All Items

def extract_all_items(assets: AssetCache, templates: t.Any) -> list[dict[str, str]]:
    """Extract all items from the asset cache."""
    items_data = []

    for template_name in ["Item", "ItemWithBoost"]:
        if template_name not in templates:
            print(f"Warning: Template '{template_name}' not found")
            continue
        
        for asset in templates[template_name].assets:
            try:
                item = {
                    "guid": asset.guid,
                    "name": get_localized_name(asset),
                    "rarity": "",
                    "trade_price": "",
                    "targets": "",
                    "buffs": "",
                    "boost_condition": "",
                    "boost_buffs": "",
                    "source": ""
                }
                
                # Get rarity
                try:
                    rarity = asset.Item.Rarity()
                    if rarity:
                        item["rarity"] = assets.properties.ui_text_cache.get_ui_text("Rarity", rarity).text()
                except:
                    pass
                
                # Get trade price
                try:
                    trade_price = asset.Item.TradePrice()
                    if trade_price:
                        item["trade_price"] = str(trade_price)
                except:
                    pass
                
                # Get targets
                try:
                    effect = asset.Effect
                    if effect:
                        target_guids = []
                        for pool in effect.Targets:
                            target_guids.extend(flatten_pool(pool.GUID()))
                        if target_guids:
                            item["targets"] = get_target_names(target_guids)
                except:
                    pass
                
                # Get buffs using buff_ui
                try:
                    effect = asset.Effect
                    if effect:
                        buff_descriptions = []
                        for buff in effect.Buffs:
                            if buff.GUID.guid:
                                buff_asset = assets[buff.GUID.guid]
                                if buff_asset:
                                    buff_desc = format_buff_attributes(buff_asset)
                                    if buff_desc and buff_desc != "No attributes":
                                        buff_descriptions.append(buff_desc)
                        if buff_descriptions:
                            item["buffs"] = " | ".join(buff_descriptions)
                except:
                    pass
                
                # Get boost condition and boost buffs
                if "ItemWithBoost" in asset.template.name:
                    try:
                        boost_condition = extract_boost_condition(asset)
                        if boost_condition:
                            item["boost_condition"] = boost_condition
                    except:
                        pass
                    
                    try:
                        boost_buffs = extract_boost_buffs(asset)
                        if boost_buffs:
                            item["boost_buffs"] = boost_buffs
                    except:
                        pass
                
                # Find sources using improved discovery
                try:
                    sources = find_all_item_sources(asset, assets)
                    sources_csv = format_all_sources_csv(sources)
                    if sources_csv:
                        item["source"] = sources_csv
                except Exception as e:
                    print(f"Error finding sources for {asset.guid}: {e}")
                    pass
                
                items_data.append(item)
                
            except Exception as e:
                print(f"Error processing asset {asset.guid}: {e}")
                continue

    print(f"Extracted {len(items_data)} items")
    return items_data


items_data = extract_all_items(assets, templates)

Extracted 391 items


In [8]:
# Preview the extracted items

def preview_items(items_data: list[dict[str, str]], num_items: int = 10) -> None:
    """Display a preview of the first N items in a formatted table."""
    df = pd.DataFrame(items_data)
    print(f"Total items: {len(df)}")
    print(f"\nColumns: {', '.join(df.columns)}")
    print(f"\nFirst {num_items} items:")
    display(df.head(num_items))


preview_items(items_data)

Total items: 391

Columns: guid, name, rarity, trade_price, targets, buffs, boost_condition, boost_buffs, source

First 10 items:


,guid,name,rarity,trade_price,targets,buffs,boost_condition,boost_buffs,source
0,71438,"Gaius Julius Lupus, Castor of Fortunes",Epic,100000,"Market, Market",Income Area Effect: +1.5,,,
1,71441,"Brutus Julius Lupus, Pollux of Polities",Epic,100000,"Libertus Residence, Plebeian Residence, Eques ...",Workforce Needed: +20%,,,
2,42057,"Abdfil, Elephant Handler",Epic,100000,"Libertus Residence, Plebeian Residence, Eques ...",Prestige: +1.5,,,
3,44431,Elephant Handler,Rare,15000,"Wheat Farm, Wheat Farm, Flax Farm, Flax Farm, ...",Prestige: +1; Productivity: +20%,,,
4,94567,"Aodhan, Master Lutist of the Scathach",Epic,100000,"Libertus Residence, Plebeian Residence, Eques ...","Happiness from Bardic Hearth, if supplied: +1;...",,,Quest: Campaign Act 2 Sidequest Captives
5,125560,Abuccus,Rare,15000,"Watchtower, Watchtower",Income Area Effect: +0.5,,,
6,95403,"Servia Bellia, Lily of the Coast",Epic,100000,"Fishing Hut, Scomber's Shack, Salt Ponds, Snai...",Income Area Effect: +1; Productivity: +25%,,,
7,96815,"Gigantulas, Polyphemian Captain",Epic,100000,"Quinquireme, Penteconter, Trireme, Flagship, R...",Hitpoints: +150%; Self-repair speed: +200%; Mo...,,,
8,96821,"Publius Quintus, Amphipraetorian",Epic,100000,"Vigiles, Vigiles, Custodia, Custodes, Medici, ...",Income Area Effect: +1; Workforce Needed: -25%...,,,
9,96819,Virtuous Volunteer,Rare,15000,"Tiler, Tiler, Concrete Mixer, Concrete Mixer, ...",Productivity: +15%; Upkeep Cost: -25%; Workfor...,,,


In [9]:
# Save to CSV

def save_items_to_csv(items_data: list[dict[str, str]], output_path: Path) -> None:
    """Save items data to a CSV file."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["guid", "name", "rarity", "trade_price", "targets", "buffs", "boost_condition", "boost_buffs", "source"])
        writer.writeheader()
        writer.writerows(items_data)
    
    print(f"Saved {len(items_data)} items to {output_path.absolute()}")


output_path = Path("results") / "tables" / "items_v8.csv"
save_items_to_csv(items_data, output_path)

Saved 391 items to c:\dev\asset-extractor\results\tables\items_v8.csv


In [10]:
# Display Statistics

def display_item_statistics(items_data: list[dict[str, str]]) -> None:
    """Display comprehensive statistics about the extracted items."""
    df = pd.DataFrame(items_data)
    
    print("\n=== Item Statistics ===")
    print(f"Total items: {len(items_data)}")

    # Count by rarity
    rarity_counts = df['rarity'].value_counts()
    print(f"\nItems by rarity:")
    for rarity, count in rarity_counts.items():
        print(f"  {rarity}: {count}")

    # Items with effects
    items_with_buffs = df[df['buffs'] != ''].shape[0]
    items_with_targets = df[df['targets'] != ''].shape[0]
    items_with_sources = df[df['source'] != ''].shape[0]
    print(f"\nItems with buffs: {items_with_buffs}")
    print(f"Items with targets: {items_with_targets}")
    print(f"Items with known sources: {items_with_sources} ({items_with_sources/len(items_data)*100:.1f}%)")
    
    # Source type breakdown
    source_type_counts = {
        "Selling": 0,
        "Research": 0,
        "Quest": 0,
        "Drops": 0,
        "Achievement": 0,
        "Festival": 0,
        "Event": 0,
        "Unlock": 0
    }
    
    for item in items_data:
        source_str = item["source"]
        if source_str:
            for source_type in source_type_counts.keys():
                if f"{source_type}:" in source_str:
                    source_type_counts[source_type] += 1
    
    print(f"\nItems by source type:")
    for source_type, count in source_type_counts.items():
        if count > 0:
            print(f"  {source_type}: {count}")

    # Price statistics
    df_with_price = df[df['trade_price'] != '']
    if len(df_with_price) > 0:
        df_with_price['trade_price_num'] = pd.to_numeric(df_with_price['trade_price'])
        print(f"\nPrice statistics (for {len(df_with_price)} items with prices):")
        print(f"  Min: {df_with_price['trade_price_num'].min()}")
        print(f"  Max: {df_with_price['trade_price_num'].max()}")
        print(f"  Average: {df_with_price['trade_price_num'].mean():.2f}")


display_item_statistics(items_data)


=== Item Statistics ===
Total items: 391

Items by rarity:
  Rare: 127
  Epic: 115
  Common: 68
  Legendary: 57
  Unique: 17
  Quest Item: 6
  Uncommon: 1

Items with buffs: 387
Items with targets: 382
Items with known sources: 338 (86.4%)

Items by source type:
  Selling: 307
  Research: 18
  Quest: 18
  Festival: 185

Price statistics (for 391 items with prices):
  Min: 100
  Max: 600000
  Average: 116039.90


In [11]:
# Display Statistics

def display_item_statistics(items_data: list[dict[str, str]]) -> None:
    """Display comprehensive statistics about the extracted items."""
    df = pd.DataFrame(items_data)
    
    print("\n=== Item Statistics ===")
    print(f"Total items: {len(items_data)}")

    # Count by rarity
    rarity_counts = df['rarity'].value_counts()
    print(f"\nItems by rarity:")
    for rarity, count in rarity_counts.items():
        print(f"  {rarity}: {count}")

    # Items with effects
    items_with_buffs = df[df['buffs'] != ''].shape[0]
    items_with_targets = df[df['targets'] != ''].shape[0]
    items_with_sources = df[df['source'] != ''].shape[0]
    print(f"\nItems with buffs: {items_with_buffs}")
    print(f"Items with targets: {items_with_targets}")
    print(f"Items with known sources: {items_with_sources}")

    # Price statistics
    df_with_price = df[df['trade_price'] != '']
    if len(df_with_price) > 0:
        df_with_price['trade_price_num'] = pd.to_numeric(df_with_price['trade_price'])
        print(f"\nPrice statistics (for {len(df_with_price)} items with prices):")
        print(f"  Min: {df_with_price['trade_price_num'].min()}")
        print(f"  Max: {df_with_price['trade_price_num'].max()}")
        print(f"  Average: {df_with_price['trade_price_num'].mean():.2f}")


display_item_statistics(items_data)


=== Item Statistics ===
Total items: 391

Items by rarity:
  Rare: 127
  Epic: 115
  Common: 68
  Legendary: 57
  Unique: 17
  Quest Item: 6
  Uncommon: 1

Items with buffs: 387
Items with targets: 382
Items with known sources: 338

Price statistics (for 391 items with prices):
  Min: 100
  Max: 600000
  Average: 116039.90
